# Build synthetic component maps (Planck 6-channel, Nside=4096)

Populate `/rds/rds-lxu/flamingo/integrated_maps_synthetic/components/` with
beam-**unconvolved** maps at the six Planck HFI frequencies from
`reference_tables/planck_info.png` (Table I):

| GHz | 100 | 143 | 217 | 353 | 545 | 857 |
|-----|-----|-----|-----|-----|-----|-----|

**Sky model:** lensed primary CMB + tSZ($\nu$) + kSZ + CIB($\nu$), all in
$\mu\mathrm{K}_\mathrm{CMB}$ at native $N_\mathrm{side}=4096$.

| Component | Source | Action |
|-----------|--------|--------|
| CMB | FLAMINGO $\kappa$ + CAMB $C_\ell^{TT}$ | simulate (pixell lensing) |
| CIB | Yang26 released bands 217/353/545/857 | copy intensity FITS; 100/143 GHz via greybody SED |
| tSZ | `lensed_tSZ_rot.fits` (Compton $y$) | copy $y$; build $\Delta T(\nu)=T_\mathrm{CMB}\,y\,f(\nu)$ |
| kSZ | `lensed_kSZ_rot.fits` (Doppler $b$) | copy $b$; $\Delta T=-T_\mathrm{CMB}\,b$ (freq.-indep.) |

Input maps: `/rds/flamingo/L2800N5040/HYDRO_FIDUCIAL/lightcone0_shells`.

Requires `pip install -e ".[cmb]"` for the lensed CMB step (`camb`, `pixell`).

In [ ]:
from pathlib import Path

from flamingo_mock import MockConfig
from flamingo_mock.config import PLANCK_FREQUENCIES_GHZ
from flamingo_mock import cib, cmb, ksz, tsz

cfg = MockConfig(
    frequencies=PLANCK_FREQUENCIES_GHZ,
    nside=4096,
    seed=42,
)
cfg.make_dirs()

OUT = cfg.out_dir / "components"
for sub in ("cmb", "cib", "tsz", "ksz"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print("data:", cfg.data_dir)
print("out: ", OUT)
print("freqs:", list(cfg.frequencies))
print("Nside:", cfg.nside)

## 1. Lensed primary CMB

In [ ]:
# ~30–60 min at Nside=4096; skipped if the FITS already exists.
cmb_uK = cmb.make_lensed_cmb(cfg, out_dir=OUT / "cmb")
print(f"CMB lensed: std={cmb_uK.std():.2f} uK")

## 2. CIB — copy released bands, approximate 100/143 GHz

Released lensed bandpass maps (217/353/545/857 GHz) are copied from the
FLAMINGO tree. **100 and 143 GHz** are outside the released set; we build
them with the three-parameter greybody SED at $z_\mathrm{eff}=1.5$ (same
method as `flamingo_mock.cib.approximate_cib_intensity`).

Note: `CIB_nonrot_BANDPASS_F143_three_params.fits` exists but is **not**
lensed — we do not use it.

In [ ]:
# Archive released intensity maps [Jy/sr] (symlink to save space)
cib.copy_released_cib_intensity(cfg, out_dir=OUT / "cib", use_symlink=True)

# Thermodynamic maps [uK_CMB] at all six frequencies
cib_uK = cib.make_cib_maps(cfg, out_dir=OUT / "cib")

## 3. tSZ — copy Compton-$y$, build $\Delta T(\nu)$

The lensed Compton-$y$ map is archived from `lensed_tSZ_rot.fits`. Per-frequency
temperature maps use the non-relativistic spectral function
$f(x)=x\coth(x/2)-4$.

In [ ]:
tsz.archive_compton_y(cfg, out_dir=OUT / "tsz", use_symlink=True)
tsz_uK = tsz.make_tsz_maps(cfg, out_dir=OUT / "tsz")

## 4. kSZ — copy Doppler-$b$, convert to $\mu\mathrm{K}_\mathrm{CMB}$

The lensed kSZ map (`lensed_kSZ_rot.fits`) stores Doppler $b$ with
$\Delta T/T_\mathrm{CMB}=-b$. The thermodynamic map is frequency independent.

In [ ]:
ksz.archive_doppler_b(cfg, out_dir=OUT / "ksz", use_symlink=True)
ksz_uK = ksz.make_ksz_map(cfg, out_dir=OUT / "ksz")
print(f"kSZ dT: std={ksz_uK.std():.3e} uK")

## 5. Inventory

In [ ]:
print(f"\nProducts under {OUT}:\n")
for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        files = sorted(sub.glob("*.fits"))
        print(f"  {sub.name}/  ({len(files)} files)")
        for p in files:
            print(f"    {p.name}  ({p.stat().st_size/1e9:.2f} GB)")